# SVG Fundamentals & LoRA Fine-Tuning Tutorial

**NYU Deep Learning — Spring 2026 Midterm Kaggle Competition**

This notebook covers:
1. What is SVG? Structure, syntax, and core concepts
2. Hands-on SVG creation — building a famous anime character
3. LoRA (Low-Rank Adaptation) theory and practical fine-tuning code

In [34]:
!pip install cairosvg

In [18]:
from IPython.display import SVG, display, HTML
import cairosvg
from IPython.display import Image
import io

---
## Part 3: LoRA Fine-Tuning — Theory

### The Problem: Full Fine-Tuning is Expensive

To adapt a pre-trained LLM (e.g., Qwen2.5-Coder with 1.5B+ parameters) for SVG generation, **full fine-tuning** updates every parameter:

$$\mathcal{L}(\theta) = \sum_{i=1}^{N} -\log P_{\theta}(y_i \mid x_i)$$

where $\theta$ represents **all** model parameters. For a 1.5B model in float32, that's ~6 GB of parameters, plus ~18 GB for optimizer states (Adam). This is often impractical.

### LoRA: Low-Rank Adaptation (Hu et al., 2021)

**Key insight:** The weight updates during fine-tuning have **low intrinsic rank**. Instead of updating the full weight matrix $W \in \mathbb{R}^{d \times k}$, we decompose the update into two low-rank matrices:

$$W' = W + \Delta W = W + BA$$

where:
- $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times k}$
- $r \ll \min(d, k)$ is the **rank** (typically 4, 8, 16, 32, or 64)
- The original $W$ is **frozen** — only $A$ and $B$ are trained

**Parameter savings:** For a linear layer $W \in \mathbb{R}^{4096 \times 4096}$:
- Full fine-tuning: $4096^2 = 16,777,216$ parameters
- LoRA with $r=16$: $(4096 \times 16) \times 2 = 131,072$ parameters (**0.78%** of original!)

### How LoRA Works — Step by Step

```
              ┌─────────────────────────┐
              │   Original Layer (Frozen)│
   x ───────►│      h = Wx              │───► h + ΔWx
      │       └─────────────────────────┘        ▲
      │                                          │
      │       ┌──────┐    ┌──────┐              │
      └──────►│  A   │───►│  B   │──── (α/r) ──┘
              │ r × k│    │ d × r│
              └──────┘    └──────┘
              Trainable low-rank adapters
```

During the forward pass:
$$h = Wx + \frac{\alpha}{r} \cdot BAx$$

where $\alpha$ is a scaling hyperparameter (often set equal to $r$ so the scaling is 1.0).

### Initialization

- $A$ is initialized with **random Gaussian** values
- $B$ is initialized to **zeros**
- This ensures $\Delta W = BA = 0$ at the start → the model begins at its pre-trained state

### Key Hyperparameters

| Parameter | Typical Values | Effect |
|-----------|---------------|--------|
| `r` (rank) | 4, 8, 16, 32, 64 | Higher = more expressive but more parameters |
| `lora_alpha` | Usually = `r` | Scaling factor; `alpha/r` controls update magnitude |
| `target_modules` | `q_proj, v_proj, k_proj, o_proj, gate_proj, up_proj, down_proj` | Which layers to adapt |
| `lora_dropout` | 0.0 – 0.1 | Regularization on LoRA layers |

### QLoRA — Quantized LoRA (Dettmers et al., 2023)

QLoRA combines LoRA with **4-bit quantization** of the base model:

1. Base model weights are quantized to **NF4** (NormalFloat 4-bit) format
2. LoRA adapters remain in **BF16/FP16** for training stability
3. Double quantization further reduces memory by quantizing the quantization constants

This allows fine-tuning a **7B model on a single 24 GB GPU** — essential for this competition!

### LoRA vs. Other PEFT Methods

| Method | Parameters Trained | Memory | Inference Cost | Merge-able? |
|--------|-------------------|--------|----------------|-------------|
| **Full Fine-Tuning** | 100% | Very High | Same | N/A |
| **LoRA** | ~0.1–1% | Low | Same (after merge) | **Yes** |
| **QLoRA** | ~0.1–1% | Very Low (4-bit base) | Same (after merge) | **Yes** |
| **Prefix Tuning** | ~0.1% | Low | Slightly higher | No |
| **Adapters** | ~1–3% | Low | Higher (extra layers) | No |
| **Prompt Tuning** | ~0.01% | Very Low | Same | No |

LoRA's key advantage: after training, the adapter weights can be **merged** back into the base model ($W' = W + BA$), resulting in **zero additional inference cost**.

---
## Part 4: LoRA Fine-Tuning — Code

Below is a practical walkthrough of setting up LoRA fine-tuning for SVG generation using HuggingFace's `peft` and `transformers` libraries.

In [19]:
# Install required packages (uncomment if needed)
!pip install transformers peft accelerate bitsandbytes datasets trl cairosvg

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
from datetime import datetime, timezone
from pathlib import Path

PROJECT_ROOT = "/content/drive/MyDrive/svg-kaggle-comptetition"
DATASET_NAME = "train_canonicalized_nosvgo_len2048.csv"
DATASET_PATH = f"{PROJECT_ROOT}/datasets/{DATASET_NAME}"

ARTIFACT_PREFIX = "qwen25coder15b_canon_nosvgo_len2048"
RUN_TAG = f"{ARTIFACT_PREFIX}_singlepass"
RUN_ID = f"{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}_{RUN_TAG}"

DRIVE_RUN_ROOT = f"{PROJECT_ROOT}/runs/{RUN_ID}"
LOCAL_RUN_ROOT = f"/content/{RUN_ID}"

DRIVE_CHECKPOINT_DIR = f"{DRIVE_RUN_ROOT}/{ARTIFACT_PREFIX}_checkpoints"
DRIVE_ADAPTER_DIR = f"{DRIVE_RUN_ROOT}/{ARTIFACT_PREFIX}_adapter"
DRIVE_MERGED_DIR = f"{DRIVE_RUN_ROOT}/{ARTIFACT_PREFIX}_merged"
DRIVE_METADATA_DIR = f"{DRIVE_RUN_ROOT}/{ARTIFACT_PREFIX}_metadata"

LOCAL_CHECKPOINT_DIR = f"{LOCAL_RUN_ROOT}/{ARTIFACT_PREFIX}_checkpoints"
LOCAL_ADAPTER_DIR = f"{LOCAL_RUN_ROOT}/{ARTIFACT_PREFIX}_adapter"
LOCAL_MERGED_DIR = f"{LOCAL_RUN_ROOT}/{ARTIFACT_PREFIX}_merged"
LOCAL_METADATA_DIR = f"{LOCAL_RUN_ROOT}/{ARTIFACT_PREFIX}_metadata"

CHECKPOINT_DIR = DRIVE_CHECKPOINT_DIR
ADAPTER_DIR = DRIVE_ADAPTER_DIR
MERGED_DIR = DRIVE_MERGED_DIR
METADATA_DIR = DRIVE_METADATA_DIR

for path in [
    DRIVE_RUN_ROOT,
    DRIVE_CHECKPOINT_DIR,
    DRIVE_ADAPTER_DIR,
    DRIVE_MERGED_DIR,
    DRIVE_METADATA_DIR,
    LOCAL_RUN_ROOT,
    LOCAL_CHECKPOINT_DIR,
    LOCAL_ADAPTER_DIR,
    LOCAL_MERGED_DIR,
    LOCAL_METADATA_DIR,
]:
    Path(path).mkdir(parents=True, exist_ok=True)

print("Run namespace ready:")
print(f"  RUN_ID: {RUN_ID}")
print(f"  DATASET_PATH: {DATASET_PATH}")
print(f"  DRIVE_CHECKPOINT_DIR: {DRIVE_CHECKPOINT_DIR}")
print(f"  DRIVE_ADAPTER_DIR: {DRIVE_ADAPTER_DIR}")
print(f"  DRIVE_MERGED_DIR: {DRIVE_MERGED_DIR}")
print(f"  DRIVE_METADATA_DIR: {DRIVE_METADATA_DIR}")
print(f"  LOCAL_CHECKPOINT_DIR: {LOCAL_CHECKPOINT_DIR}")
print(f"  LOCAL_ADAPTER_DIR: {LOCAL_ADAPTER_DIR}")
print(f"  LOCAL_MERGED_DIR: {LOCAL_MERGED_DIR}")
print(f"  LOCAL_METADATA_DIR: {LOCAL_METADATA_DIR}")


Run namespace ready:
  RUN_ID: 20260401T215406Z_qwen25coder15b_canon_nosvgo_len2048_singlepass
  DATASET_PATH: /content/drive/MyDrive/svg-kaggle-comptetition/datasets/train_canonicalized_nosvgo_len2048.csv
  DRIVE_CHECKPOINT_DIR: /content/drive/MyDrive/svg-kaggle-comptetition/runs/20260401T215406Z_qwen25coder15b_canon_nosvgo_len2048_singlepass/qwen25coder15b_canon_nosvgo_len2048_checkpoints
  DRIVE_ADAPTER_DIR: /content/drive/MyDrive/svg-kaggle-comptetition/runs/20260401T215406Z_qwen25coder15b_canon_nosvgo_len2048_singlepass/qwen25coder15b_canon_nosvgo_len2048_adapter
  DRIVE_MERGED_DIR: /content/drive/MyDrive/svg-kaggle-comptetition/runs/20260401T215406Z_qwen25coder15b_canon_nosvgo_len2048_singlepass/qwen25coder15b_canon_nosvgo_len2048_merged
  DRIVE_METADATA_DIR: /content/drive/MyDrive/svg-kaggle-comptetition/runs/20260401T215406Z_qwen25coder15b_canon_nosvgo_len2048_singlepass/qwen25coder15b_canon_nosvgo_len2048_metadata
  LOCAL_CHECKPOINT_DIR: /content/20260401T215406Z_qwen25coder15

In [22]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# ---------- 1. Model & Tokenizer ----------
MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"  # Good balance of size and capability

# QLoRA: load base model in 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",             # NormalFloat 4-bit
    bnb_4bit_compute_dtype=torch.bfloat16,  # Compute in BF16
    bnb_4bit_use_double_quant=True,         # Double quantization
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

# Prepare model for k-bit training (freezes base, enables gradient checkpointing)
model = prepare_model_for_kbit_training(model)

print(f"Model loaded: {MODEL_ID}")
print(f"Model dtype: {model.dtype}")
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen2.5-Coder-1.5B-Instruct
Model dtype: torch.float32
Total parameters: 888,616,448


In [23]:
# ---------- 2. LoRA Configuration ----------
lora_config = LoraConfig(
    r=16,                     # Rank — start with 16, increase if underfitting
    lora_alpha=32,            # Scaling: effective lr multiplier = alpha/r = 2.0
    lora_dropout=0.05,        # Light regularization
    target_modules=[          # Which linear layers to add LoRA to
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention
        "gate_proj", "up_proj", "down_proj",      # MLP (feed-forward)
    ],
    bias="none",              # Don't train biases
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Print trainable vs. total parameters
model.print_trainable_parameters()
# Output example: "trainable params: 6,815,744 || all params: 1,549,507,584 || trainable%: 0.4399"

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [24]:
# ---------- 3. Inspecting LoRA Layers ----------
# Let's see what LoRA actually added to the model

print("Layers with LoRA adapters:\n")
for name, module in model.named_modules():
    if "lora" in name.lower() and hasattr(module, 'weight'):
        print(f"  {name}: {module.weight.shape}")

Layers with LoRA adapters:

  base_model.model.model.layers.0.self_attn.q_proj.lora_A.default: torch.Size([16, 1536])
  base_model.model.model.layers.0.self_attn.q_proj.lora_B.default: torch.Size([1536, 16])
  base_model.model.model.layers.0.self_attn.k_proj.lora_A.default: torch.Size([16, 1536])
  base_model.model.model.layers.0.self_attn.k_proj.lora_B.default: torch.Size([256, 16])
  base_model.model.model.layers.0.self_attn.v_proj.lora_A.default: torch.Size([16, 1536])
  base_model.model.model.layers.0.self_attn.v_proj.lora_B.default: torch.Size([256, 16])
  base_model.model.model.layers.0.self_attn.o_proj.lora_A.default: torch.Size([16, 1536])
  base_model.model.model.layers.0.self_attn.o_proj.lora_B.default: torch.Size([1536, 16])
  base_model.model.model.layers.0.mlp.gate_proj.lora_A.default: torch.Size([16, 1536])
  base_model.model.model.layers.0.mlp.gate_proj.lora_B.default: torch.Size([8960, 16])
  base_model.model.model.layers.0.mlp.up_proj.lora_A.default: torch.Size([16, 15

In [25]:
# ---------- 4. Dataset Preparation ----------
# For SVG fine-tuning, each sample is a (prompt, SVG_code) pair formatted for instruction tuning

# def format_svg_sample(prompt: str, svg_code: str) -> str:
#     """Format a prompt-SVG pair into the chat template expected by the model."""
#     messages = [
#         {"role": "system", "content": "You are an expert SVG code generator. Generate clean, valid SVG code based on the user's description."},
#         {"role": "user", "content": prompt},
#         {"role": "assistant", "content": svg_code},
#     ]
#     return tokenizer.apply_chat_template(messages, tokenize=False)

def format_svg_sample(prompt: str, svg_code: str) -> str:
    return f"Prompt: {prompt}\nSVG:\n{svg_code}"

# Example using the strict training root format
sample = format_svg_sample(
    prompt="A red circle centered on a white background",
    svg_code='<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><circle cx="128" cy="128" r="96" fill="red"/></svg>'
)
print("Formatted training sample:")
print(sample[:500])

Formatted training sample:
Prompt: A red circle centered on a white background
SVG:
<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><circle cx="128" cy="128" r="96" fill="red"/></svg>


In [26]:
# ---------- 5. Training Setup (using SFTTrainer from TRL) ----------
from transformers import TrainingArguments

# These are reference training arguments — adjust based on your GPU and dataset size
# training_args = TrainingArguments(
#     output_dir="./svg-lora-checkpoints",
#     num_train_epochs=3,
#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=8,       # Effective batch size = 2 * 8 = 16
#     learning_rate=1e-4,                  # LoRA typically uses higher LR than full FT
#     lr_scheduler_type="cosine",
#     warmup_ratio=0.05,
#     bf16=True,                           # Use BF16 mixed precision
#     logging_steps=10,
#     save_strategy="steps",
#     save_steps=200,
#     eval_strategy="steps",
#     eval_steps=200,
#     optim="paged_adamw_8bit",            # Memory-efficient optimizer
#     gradient_checkpointing=True,         # Trade compute for memory
#     max_grad_norm=0.3,
#     report_to="none",                    # Set to "wandb" for experiment tracking
# )

# print("Training arguments configured.")
# print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")

from trl import SFTConfig, SFTTrainer

sft_args = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=50,
    save_strategy="steps",
    save_steps=1000,
    eval_strategy="no",
    optim="paged_adamw_8bit",
    gradient_checkpointing=False,
    max_grad_norm=0.3,
    report_to="none",
    max_length=2048,
    packing=False,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [27]:
import csv
import json
from pathlib import Path

expected_dataset_path = "/content/drive/MyDrive/svg-kaggle-comptetition/datasets/train_canonicalized_nosvgo_len2048.csv"
dataset_file = Path(DATASET_PATH)

if not dataset_file.exists():
    raise FileNotFoundError(
        f"Missing canonicalized dataset at {expected_dataset_path}. "
        "Generate or upload train_canonicalized_nosvgo_len2048.csv before starting training."
    )

with dataset_file.open("r", encoding="utf-8", newline="") as handle:
    reader = csv.reader(handle)
    column_names = next(reader, None)
    if column_names != ["prompt", "svg"]:
        raise ValueError(f"Expected CSV columns ['prompt', 'svg'], found {column_names}")
    row_count = sum(1 for _ in reader)

baseline_paths = {
    "checkpoints": f"{PROJECT_ROOT}/svg-lora-checkpoints",
    "adapter": f"{PROJECT_ROOT}/svg-lora-adapter",
    "merged": f"{PROJECT_ROOT}/svg-model-merged",
}


def size_in_bytes(path_str: str):
    path = Path(path_str)
    if not path.exists():
        return None
    if path.is_file():
        return path.stat().st_size
    return sum(item.stat().st_size for item in path.rglob("*") if item.is_file())

run_config = {
    "run_id": RUN_ID,
    "run_tag": RUN_TAG,
    "artifact_prefix": ARTIFACT_PREFIX,
    "dataset_path": DATASET_PATH,
    "row_count": row_count,
    "column_names": column_names,
    "filter_rule": "prompt and svg must both be non-null and non-empty after strip()",
    "train_eval_split_seed": 42,
    "text_format": "Prompt: {prompt}\nSVG:\n{svg}",
    "drive_paths": {
        "checkpoints": DRIVE_CHECKPOINT_DIR,
        "adapter": DRIVE_ADAPTER_DIR,
        "merged": DRIVE_MERGED_DIR,
        "metadata": DRIVE_METADATA_DIR,
    },
    "local_paths": {
        "checkpoints": LOCAL_CHECKPOINT_DIR,
        "adapter": LOCAL_ADAPTER_DIR,
        "merged": LOCAL_MERGED_DIR,
        "metadata": LOCAL_METADATA_DIR,
    },
}

baseline_reference = {
    "run_id": RUN_ID,
    "source_notebook": "archive/notebooks/training/main.ipynb",
    "baseline_training_settings": {
        "epochs": 1,
        "eval_strategy": "no",
        "max_length": 1024,
        "per_device_train_batch_size": 4,
        "gradient_accumulation_steps": 4,
    },
    "baseline_known_limitations": [
        "no canonicalization",
        "max_length=1024 may truncate long SVG samples",
    ],
    "artifacts": {
        name: {
            "path": path,
            "exists": Path(path).exists(),
            "size_bytes": size_in_bytes(path),
        }
        for name, path in baseline_paths.items()
    },
}

drive_run_config_path = Path(DRIVE_METADATA_DIR) / "run_config.json"
local_run_config_path = Path(LOCAL_METADATA_DIR) / "run_config.json"
drive_baseline_reference_path = Path(DRIVE_METADATA_DIR) / "baseline_reference.json"
local_baseline_reference_path = Path(LOCAL_METADATA_DIR) / "baseline_reference.json"

for output_path, payload in [
    (drive_run_config_path, run_config),
    (local_run_config_path, run_config),
    (drive_baseline_reference_path, baseline_reference),
    (local_baseline_reference_path, baseline_reference),
]:
    output_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

print(f"Canonicalized dataset path: {DATASET_PATH}")
print(f"Canonicalized dataset rows: {row_count:,}")
print(f"Dataset columns: {column_names}")
print(f"Wrote run config to Drive: {drive_run_config_path}")
print(f"Wrote run config locally: {local_run_config_path}")
print(f"Wrote baseline snapshot to Drive: {drive_baseline_reference_path}")
print(f"Wrote baseline snapshot locally: {local_baseline_reference_path}")


Canonicalized dataset path: /content/drive/MyDrive/svg-kaggle-comptetition/datasets/train_canonicalized_nosvgo_len2048.csv
Canonicalized dataset rows: 39,298
Dataset columns: ['prompt', 'svg']
Wrote run config to Drive: /content/drive/MyDrive/svg-kaggle-comptetition/runs/20260401T215406Z_qwen25coder15b_canon_nosvgo_len2048_singlepass/qwen25coder15b_canon_nosvgo_len2048_metadata/run_config.json
Wrote run config locally: /content/20260401T215406Z_qwen25coder15b_canon_nosvgo_len2048_singlepass/qwen25coder15b_canon_nosvgo_len2048_metadata/run_config.json
Wrote baseline snapshot to Drive: /content/drive/MyDrive/svg-kaggle-comptetition/runs/20260401T215406Z_qwen25coder15b_canon_nosvgo_len2048_singlepass/qwen25coder15b_canon_nosvgo_len2048_metadata/baseline_reference.json
Wrote baseline snapshot locally: /content/20260401T215406Z_qwen25coder15b_canon_nosvgo_len2048_singlepass/qwen25coder15b_canon_nosvgo_len2048_metadata/baseline_reference.json


In [28]:
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

csv_path = DATASET_PATH
raw_dataset = load_dataset("csv", data_files=csv_path)["train"]

def is_valid_row(example):
    return (
        example.get("prompt") is not None
        and example.get("svg") is not None
        and str(example["prompt"]).strip() != ""
        and str(example["svg"]).strip() != ""
    )

def to_training_text(example):
    return {
        "text": format_svg_sample(
            prompt=str(example["prompt"]).strip(),
            svg_code=str(example["svg"]).strip()
        )
    }

def find_subsequence(sequence, subsequence):
    max_start = len(sequence) - len(subsequence)
    for start in range(max_start + 1):
        if sequence[start:start + len(subsequence)] == subsequence:
            return start
    return -1

class SvgOnlyLossCollator:
    def __init__(self, tokenizer, response_template: str, max_length: int):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.response_template_ids = tokenizer.encode(response_template, add_special_tokens=False)

    def __call__(self, features):
        if "input_ids" in features[0]:
            batch = self.tokenizer.pad(features, padding=True, return_tensors="pt")
        else:
            texts = [feature["text"] for feature in features]
            batch = self.tokenizer(
                texts,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors="pt",
            )

        labels = batch["input_ids"].clone()
        labels[batch["attention_mask"] == 0] = -100

        for row_idx, input_ids in enumerate(batch["input_ids"].tolist()):
            start = find_subsequence(input_ids, self.response_template_ids)
            if start == -1:
                raise ValueError("Could not find the SVG response template in a training sample.")
            labels[row_idx, : start + len(self.response_template_ids)] = -100

        batch["labels"] = labels
        return batch

dataset = raw_dataset.filter(is_valid_row)
dataset = dataset.map(to_training_text, remove_columns=raw_dataset.column_names)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]
svg_only_loss_collator = SvgOnlyLossCollator(tokenizer, response_template="SVG:\n", max_length=sft_args.max_length)

print(train_dataset[0]["text"][:1000])
print("SVG-only loss masking enabled after the response template 'SVG:\\n'.")

# Optional length sanity check
sample_count = min(100, len(train_dataset))
lengths = [
    len(tokenizer(train_dataset[i]["text"])["input_ids"])
    for i in range(sample_count)
]
print("Max token length:", max(lengths))
print("Avg token length:", sum(lengths) / len(lengths))

Prompt: A black rectangular outline contains two vertical white rectangles; one is narrower and positioned to the left, while the other is wider and positioned to the right, both centered within the black outline against a white background.
SVG:
<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><path fill="currentColor" fill-opacity="1" d="M175 16.667 L25 16.667 A80.333 8.333 0 0 0 16.667 25 L16.667 175 A80.333 8.333 0 0 0 25 183.333 L175 183.333 A80.333 8.333 0 0 0 183.333 175 L183.333 25 A80.333 8.333 0 0 0 175 16.667 Z M66.667 166.667 L33.333 166.667 L33.333 33.333 L66.667 33.333 L66.667 166.667 Z M166.667 166.667 L83.333 166.667 L83.333 33.333 L166.667 33.333 L166.667 166.667 Z"></path></svg>
SVG-only loss masking enabled after the response template 'SVG:\n'.
Max token length: 2007
Avg token length: 924.32


In [29]:
# ---------- 6. Training Loop (reference — requires a dataset) ----------
# Uncomment and adapt once you have your SVG dataset loaded

import shutil
from pathlib import Path

from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_dataset, # Your HuggingFace Dataset object
    eval_dataset=eval_dataset,
    processing_class=tokenizer, # SVGs can be long — adjust as needed
    data_collator=svg_only_loss_collator,
)

trainer.train()

# Save the LoRA adapter (small file — typically 10-50 MB)
model.save_pretrained(DRIVE_ADAPTER_DIR)
tokenizer.save_pretrained(DRIVE_ADAPTER_DIR)
model.save_pretrained(LOCAL_ADAPTER_DIR)
tokenizer.save_pretrained(LOCAL_ADAPTER_DIR)

if Path(DRIVE_CHECKPOINT_DIR).exists() and any(Path(DRIVE_CHECKPOINT_DIR).iterdir()):
    shutil.rmtree(LOCAL_CHECKPOINT_DIR, ignore_errors=True)
    shutil.copytree(DRIVE_CHECKPOINT_DIR, LOCAL_CHECKPOINT_DIR, dirs_exist_ok=True)
    print(f"Checkpoints mirrored locally: {LOCAL_CHECKPOINT_DIR}")
else:
    print("No checkpoint directories were emitted during training.")

print(f"Adapter saved to Drive: {DRIVE_ADAPTER_DIR}")
print(f"Adapter saved locally: {LOCAL_ADAPTER_DIR}")

Adding EOS to train dataset:   0%|          | 0/35368 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/35368 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/3930 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/3930 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
50,0.728637
100,0.631689
150,0.581318
200,0.573496
250,0.562681
300,0.552713
350,0.539150
400,0.519601
450,0.518849
500,0.521832


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Checkpoints mirrored locally: /content/20260401T215406Z_qwen25coder15b_canon_nosvgo_len2048_singlepass/qwen25coder15b_canon_nosvgo_len2048_checkpoints
Adapter saved to Drive: /content/drive/MyDrive/svg-kaggle-comptetition/runs/20260401T215406Z_qwen25coder15b_canon_nosvgo_len2048_singlepass/qwen25coder15b_canon_nosvgo_len2048_adapter
Adapter saved locally: /content/20260401T215406Z_qwen25coder15b_canon_nosvgo_len2048_singlepass/qwen25coder15b_canon_nosvgo_len2048_adapter


In [30]:
print("pad:", tokenizer.pad_token, tokenizer.pad_token_id)
print("bos:", tokenizer.bos_token, tokenizer.bos_token_id)
print("eos:", tokenizer.eos_token, tokenizer.eos_token_id)

print("model pad:", model.config.pad_token_id)

pad: <|endoftext|> 151643
bos: None None
eos: <|im_end|> 151645
model pad: 151643


In [31]:
# ---------- 7. Loading and Merging LoRA Weights ----------
# After training, load the adapter, merge it into the base model, and save the merged model.

import json
from datetime import datetime, timezone
from pathlib import Path

from peft import PeftModel

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto")
model = PeftModel.from_pretrained(model, DRIVE_ADAPTER_DIR)
model = model.merge_and_unload()

model.save_pretrained(DRIVE_MERGED_DIR)
tokenizer.save_pretrained(DRIVE_MERGED_DIR)
model.save_pretrained(LOCAL_MERGED_DIR)
tokenizer.save_pretrained(LOCAL_MERGED_DIR)

checkpoint_paths = sorted(
    [path for path in Path(DRIVE_CHECKPOINT_DIR).glob("checkpoint-*") if path.is_dir()],
    key=lambda path: int(path.name.split("-")[-1]),
)
final_checkpoint_path = str(checkpoint_paths[-1]) if checkpoint_paths else None

training_config = {
    "output_dir": sft_args.output_dir,
    "num_train_epochs": sft_args.num_train_epochs,
    "per_device_train_batch_size": sft_args.per_device_train_batch_size,
    "gradient_accumulation_steps": sft_args.gradient_accumulation_steps,
    "learning_rate": sft_args.learning_rate,
    "lr_scheduler_type": sft_args.lr_scheduler_type,
    "warmup_ratio": sft_args.warmup_ratio,
    "bf16": sft_args.bf16,
    "save_strategy": sft_args.save_strategy,
    "save_steps": sft_args.save_steps,
    "eval_strategy": sft_args.eval_strategy,
    "gradient_checkpointing": sft_args.gradient_checkpointing,
    "max_length": sft_args.max_length,
    "packing": sft_args.packing,
}

run_summary = {
    "run_id": RUN_ID,
    "run_tag": RUN_TAG,
    "artifact_prefix": ARTIFACT_PREFIX,
    "completion_timestamp_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "drive_paths": {
        "checkpoints": DRIVE_CHECKPOINT_DIR,
        "adapter": DRIVE_ADAPTER_DIR,
        "merged": DRIVE_MERGED_DIR,
        "metadata": DRIVE_METADATA_DIR,
    },
    "local_paths": {
        "checkpoints": LOCAL_CHECKPOINT_DIR,
        "adapter": LOCAL_ADAPTER_DIR,
        "merged": LOCAL_MERGED_DIR,
        "metadata": LOCAL_METADATA_DIR,
    },
    "training_config": training_config,
    "trainer_global_step": int(trainer.state.global_step),
    "final_checkpoint_path": final_checkpoint_path,
}

drive_run_summary_path = Path(DRIVE_METADATA_DIR) / "run_summary.json"
local_run_summary_path = Path(LOCAL_METADATA_DIR) / "run_summary.json"

for output_path in [drive_run_summary_path, local_run_summary_path]:
    output_path.write_text(json.dumps(run_summary, indent=2), encoding="utf-8")

print(f"Merged model saved to Drive: {DRIVE_MERGED_DIR}")
print(f"Merged model saved locally: {LOCAL_MERGED_DIR}")
print(f"Run summary written to Drive: {drive_run_summary_path}")
print(f"Run summary written locally: {local_run_summary_path}")


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to Drive: /content/drive/MyDrive/svg-kaggle-comptetition/runs/20260401T215406Z_qwen25coder15b_canon_nosvgo_len2048_singlepass/qwen25coder15b_canon_nosvgo_len2048_merged
Merged model saved locally: /content/20260401T215406Z_qwen25coder15b_canon_nosvgo_len2048_singlepass/qwen25coder15b_canon_nosvgo_len2048_merged
Run summary written to Drive: /content/drive/MyDrive/svg-kaggle-comptetition/runs/20260401T215406Z_qwen25coder15b_canon_nosvgo_len2048_singlepass/qwen25coder15b_canon_nosvgo_len2048_metadata/run_summary.json
Run summary written locally: /content/20260401T215406Z_qwen25coder15b_canon_nosvgo_len2048_singlepass/qwen25coder15b_canon_nosvgo_len2048_metadata/run_summary.json


In [32]:
model.eval()

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [33]:
import re
import xml.etree.ElementTree as ET

ALLOWED_TAGS = {
    "svg","g","path","rect","circle","ellipse",
    "line","polyline","polygon","defs","use",
    "symbol","clipPath","mask","linearGradient",
    "radialGradient","stop","text","tspan",
    "title","desc","style","pattern","marker","filter"
}

FALLBACK_SVG = '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 256 256"></svg>'


def extract_svg(text: str) -> str:
    match = re.search(r"<svg\b[^>]*>.*?</svg>", text, flags=re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(0).strip()
    return text.strip()




def is_valid_svg_basic(svg: str) -> bool:
    if not svg:
        return False

    if len(svg) > 16000:
        return False

    try:
        root = ET.fromstring(svg)
    except Exception:
        return False

    root_tag = root.tag.split("}")[-1]
    if root_tag != "svg":
        return False

    path_count = 0
    for elem in root.iter():
        tag = elem.tag.split("}")[-1]
        if tag not in ALLOWED_TAGS:
            return False
        if tag == "path":
            path_count += 1

    if path_count > 256:
        return False

    return True


def clean_svg_output(raw_text: str) -> str:
    svg = extract_svg(raw_text)
    if not is_valid_svg_basic(svg):
        return FALLBACK_SVG
    return svg


SYSTEM_PROMPT = (
    "You are an expert SVG code generator. "
    "Return only valid SVG markup. "
    "Do not include explanations, markdown, backticks, or any extra text."
)


def build_model_input_chat(prompt: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

def build_model_input_plain(prompt: str) -> str:
    return f"Prompt: {prompt.strip()}\nSVG:\n"


def generate_svg(
    prompt: str,
    max_new_tokens: int = 1024,
    do_sample: bool = False,
    temperature: float = 0.2,
    use_chat_template: bool = False,
) -> str:

    if use_chat_template:
        input_text = build_model_input_chat(prompt)
    else:
        input_text = build_model_input_plain(prompt)

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    svg = clean_svg_output(decoded)
    return svg

---
## Summary

| Topic | Key Takeaway |
|-------|-------------|
| **SVG** | XML-based vector format; infinite scalability; LLMs can generate it as code tokens |
| **SVG Primitives** | `rect`, `circle`, `ellipse`, `path` (Bézier curves), `polygon`, `text`, `g` (groups) |
| **Rendering** | `cairosvg` converts SVG → PNG for evaluation; `IPython.display.SVG` for inline display |
| **LoRA** | Decomposes weight update as $\Delta W = BA$ with rank $r \ll d$; trains <1% of parameters |
| **QLoRA** | 4-bit NF4 quantized base + BF16 LoRA adapters → fine-tune large models on consumer GPUs |
| **Merging** | After training: `merge_and_unload()` folds adapters back → zero inference overhead |

### Next Steps
1. Explore the competition training data in `competition_assets/datasets/svg-train-public/`
2. Prepare your dataset using the `format_svg_sample()` function above
3. Fine-tune with QLoRA and experiment with different `r`, `target_modules`, and learning rates
4. Evaluate outputs using the competition metric (see `competition_assets/evaluation/`)